In [ ]:
# ==========================================
# PHASE 05.5 - STEP 00
# Generate Thesis Figures from Phase 05 Results
# (UPDATED & CORRECTED)
# ==========================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# 2. SETUP PATHS
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
METRICS_PATH = os.path.join(BASE_PATH, "Results", "Metrics", "phase05_classification_report.csv")
FIGURES_DIR = os.path.join(BASE_PATH, "Results", "Figures")

os.makedirs(FIGURES_DIR, exist_ok=True)

print("🚀 PHASE 05.5 - STEP 00 STARTED...")

# 3. LOAD DATA
if not os.path.exists(METRICS_PATH):
    raise FileNotFoundError(f"❌ Report not found! Please check: {METRICS_PATH}")

df = pd.read_csv(METRICS_PATH, index_col=0)

# 4. PREPARE DATA FOR PLOTTING
classes = ["NoFall", "Faint", "Slip", "Trip"]
class_data = df.loc[classes].reset_index().rename(columns={"index": "Class"})

df_melted = class_data.melt(
    id_vars="Class",
    value_vars=["precision", "recall", "f1-score"],
    var_name="Metric",
    value_name="Score"
)

df_melted["Metric"] = df_melted["Metric"].replace({
    "precision": "Precision",
    "recall": "Recall",
    "f1-score": "F1-Score"
})

# 5. FIGURE 1: PER-CLASS PERFORMANCE (FIXED TITLE)
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
ax = sns.barplot(
    data=df_melted,
    x="Class",
    y="Score",
    hue="Metric",
    palette="viridis"
)

plt.title(
    "Per-Class Performance (Enhanced Bi-LSTM Model)",
    fontsize=14,
    fontweight="bold"
)
plt.ylim(0, 1.1)
plt.ylabel("Score (0.0 - 1.0)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

save_path_1 = os.path.join(FIGURES_DIR, "phase05_per_class_metrics.png")
plt.tight_layout()
plt.savefig(save_path_1, dpi=300)
plt.show()

# 6. FIGURE 2: GLOBAL SUMMARY
accuracy = df.loc["accuracy"].values[0]
macro_f1 = df.loc["macro avg", "f1-score"]
weighted_f1 = df.loc["weighted avg", "f1-score"]

summary_data = pd.DataFrame({
    "Metric": ["Overall Accuracy", "Macro F1-Score", "Weighted F1-Score"],
    "Score": [accuracy, macro_f1, weighted_f1]
})

plt.figure(figsize=(8, 5))
ax2 = sns.barplot(
    data=summary_data,
    x="Metric",
    y="Score",
    palette="magma",
    hue="Metric",
    legend=False
)

plt.title("Global Model Performance Summary", fontsize=14, fontweight="bold")
plt.ylim(0, 1.1)

for container in ax2.containers:
    ax2.bar_label(container, fmt="%.3f", padding=3, fontweight="bold")

save_path_2 = os.path.join(FIGURES_DIR, "phase05_global_summary.png")
plt.tight_layout()
plt.savefig(save_path_2, dpi=300)
plt.show()

print("\n✅ PHASE 05.5 - STEP 00 COMPLETE")
print(f"📁 Figures saved in: {FIGURES_DIR}")

In [ ]:
# ==============================
# PHASE 05.5 - STEP 5.5-01
# Inference Preprocessing Alignment (MANDATORY)
# ==============================

# Install ultralytics if not already installed
!pip install ultralytics

import os
import cv2
import numpy as np
from ultralytics import YOLO

# 1. PATHS (DRIVE ALREADY MOUNTED)
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
TEST_VIDEO_DIR = os.path.join(BASE_PATH, "GUB-STFN-Fall-Dataset", "Slip")
RESULT_PATH = os.path.join(BASE_PATH, "Results", "Inference_Debug")

os.makedirs(RESULT_PATH, exist_ok=True)

# 2. CONFIG (MUST MATCH PHASE 03)
SEQ_LEN = 50
STRIDE = 10
YOLO_MODEL = "yolov8n-pose.pt"

# ==============================
# PREPROCESSING FUNCTIONS
# ==============================

pose_model = YOLO(YOLO_MODEL)

def extract_raw_skeletons(video_path):
    cap = cv2.VideoCapture(video_path)
    skeletons = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = pose_model(frame, verbose=False)
        skel = np.zeros((17, 3))

        if results and results[0].keypoints is not None:
            kp = results[0].keypoints.xy.cpu().numpy()
            conf = results[0].keypoints.conf.cpu().numpy()

            if len(kp) > 0:
                best_idx = np.argmax(np.mean(conf, axis=1))
                skel = np.concatenate(
                    [kp[best_idx], conf[best_idx][:, None]], axis=1
                )

        skeletons.append(skel)

    cap.release()
    return np.array(skeletons)

def hip_center_normalize(data):
    xy = data[..., :2]
    conf = data[..., 2:]

    hip_center = (xy[:, 11:12, :] + xy[:, 12:13, :]) / 2
    xy_norm = xy - hip_center

    return np.concatenate([xy_norm, conf], axis=2)

def build_sequences(norm_data):
    sequences = []

    for i in range(0, len(norm_data) - SEQ_LEN, STRIDE):
        window = norm_data[i:i + SEQ_LEN]
        sequences.append(window.reshape(SEQ_LEN, -1))

    return np.array(sequences)

# ==============================
# EXECUTION
# ==============================

video_files = [f for f in os.listdir(TEST_VIDEO_DIR) if f.endswith(".mp4")]
if not video_files:
    raise FileNotFoundError("No test video found for Phase 5.5.")

video_path = os.path.join(TEST_VIDEO_DIR, video_files[0])

raw_skeletons = extract_raw_skeletons(video_path)

if len(raw_skeletons) < SEQ_LEN:
    print("❌ Video too short for sequence alignment.")
else:
    normalized = hip_center_normalize(raw_skeletons)
    aligned_input = build_sequences(normalized)

    print("Raw Skeleton Shape :", raw_skeletons.shape)
    print("Normalized Shape   :", normalized.shape)
    print("Final Input Shape  :", aligned_input.shape)

    np.save(
        os.path.join(RESULT_PATH, "inference_ready_sample.npy"),
        aligned_input
    )

    print("\n✅ PHASE 05.5 – STEP 5.5-01 COMPLETE")
    print("Inference preprocessing matches training pipeline.")

In [ ]:
# ==============================
# PHASE 05.5 - STEP 5.5-02
# Prediction Stability & Majority Voting
# ==============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from collections import Counter

# 2. PATHS
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
# NOTE: We use the Enhanced model as it is usually the 'Thesis Hero'
MODEL_PATH = os.path.join(BASE_PATH, "Models", "model_enhanced.keras")
INPUT_PATH = os.path.join(BASE_PATH, "Results", "Inference_Debug", "inference_ready_sample.npy")
RESULT_DIR = os.path.join(BASE_PATH, "Results", "Inference_Debug")
FIG_DIR = os.path.join(BASE_PATH, "Results", "Figures") # Saving figure here too

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# 3. CLASS NAMES (Must match training order EXACTLY)
CLASSES = ["NoFall", "Faint", "Slip", "Trip"]

# ==============================
# LOAD DATA & MODEL
# ==============================

print(f"[1/4] Loading Input Data...")
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"❌ Input file not found: {INPUT_PATH}\n   Run Step 5.5-01 first.")

# Load the (N, 50, 51) array from Step 01
X_input = np.load(INPUT_PATH)
print(f"   Input Shape: {X_input.shape}")

print(f"\n[2/4] Loading Frozen Model (v1.0)...")
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("   ✅ Model Loaded Successfully.")
except Exception as e:
    print(f"⚠️ Warning: Enhanced model not found. Trying Baseline...")
    MODEL_PATH = os.path.join(BASE_PATH, "Models", "model_baseline.keras")
    model = tf.keras.models.load_model(MODEL_PATH)
    print("   ✅ Baseline Model Loaded.")

# ==============================
# INFERENCE & VOTING
# ==============================

print(f"\n[3/4] Running Inference...")
# 1. Predict probabilities for all sequences
probs = model.predict(X_input, verbose=1)

# 2. Convert to Class IDs
seq_preds = np.argmax(probs, axis=1)
seq_labels = [CLASSES[i] for i in seq_preds]

# 3. Majority Voting
vote_counts = Counter(seq_labels)
# Get the winner
final_class = vote_counts.most_common(1)[0][0]
# Calculate simple confidence (Votes for Winner / Total Votes)
confidence = vote_counts[final_class] / len(seq_labels) * 100

print("\n" + "="*30)
print(f"🎯 FINAL PREDICTION: {final_class}")
print(f"   Confidence: {confidence:.1f}% ({vote_counts[final_class]}/{len(seq_labels)} votes)")
print("="*30)
print(f"   Full Sequence Votes: {seq_labels}")

# ==============================
# VISUALIZATION & LOGGING
# ==============================

print(f"\n[4/4] Saving Results...")

# 1. Generate Vote Distribution Chart
plt.figure(figsize=(8, 5))
# Prepare data for plotting
all_classes_counts = {cls: vote_counts.get(cls, 0) for cls in CLASSES}

sns.barplot(x=list(all_classes_counts.keys()), y=list(all_classes_counts.values()), palette="viridis")
plt.title(f"Inference Vote Distribution\nFinal Decision: {final_class}")
plt.xlabel("Class")
plt.ylabel("Number of Sequence Votes")
plt.ylim(0, len(seq_labels) + 2) # Add headroom

# Save Figure to Debug folder AND Figures folder
fig_path_debug = os.path.join(RESULT_DIR, "prediction_vote_distribution.png")
fig_path_thesis = os.path.join(FIG_DIR, "phase5_5_inference_vote.png")
plt.savefig(fig_path_debug, dpi=300)
plt.savefig(fig_path_thesis, dpi=300)
plt.show()
print(f"   📊 Figure Saved: {fig_path_thesis}")

# 2. Save CSV Log
log_data = {
    "Total_Sequences": [len(seq_labels)],
    "Final_Prediction": [final_class],
    "Confidence_Percent": [round(confidence, 2)],
    "Votes_Breakdown": [str(dict(vote_counts))]
}
csv_path = os.path.join(RESULT_DIR, "phase_5_5_inference_log.csv")
pd.DataFrame(log_data).to_csv(csv_path, index=False)
print(f"   📝 Log Saved: {csv_path}")

print("\n✅ PHASE 05.5 – STEP 02 COMPLETE")
print("Input aligned ✔")
print("Output stabilized ✔")
print("Ready for Phase 6 (Demo / Alarm Logic)")